<a href="https://colab.research.google.com/github/safdarmarwat/colab/blob/main/testing_RAG02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-groq langchain-community pypdf sentence-transformers faiss-cpu gradio

In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

# 1. Load and Chunk
def process_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # Split text into 1000-character chunks with a bit of overlap
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    # 2. Create Embeddings & Store in FAISS
    # This model runs locally on the CPU (free!)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
    return vectorstore.as_retriever()

# 3. Setup LLM (Groq)
llm = ChatGroq(model_name="llama3-8b-8192", temperature=0)

# 4. Create the RAG Chain
def get_response(retriever, query):
    qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    return qa_chain.invoke(query)["result"]

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [5]:
!pip install -q langchain-text-splitters langchain-huggingface

In [6]:
from langchain_community.document_loaders import PyPDFLoader
# Note the 's' at the end of text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

In [7]:
import gradio as gr

# Global variable to store the retriever once a file is uploaded
retriever = None

def upload_and_chat(file, history, message):
    global retriever
    if file is not None and retriever is None:
        retriever = process_pdf(file.name)

    if retriever is None:
        return history + [("Please upload a PDF first!", None)]

    response = get_response(retriever, message)
    history.append((message, response))
    return history, ""

with gr.Blocks() as demo:
    gr.Markdown("# 🤖 Simple RAG Chatbot")
    file_input = gr.File(label="Upload PDF")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask a question about your document")
    clear = gr.Button("Clear")

    msg.submit(upload_and_chat, [file_input, chatbot, msg], [chatbot, msg])
    clear.click(lambda: None, None, chatbot, queue=False)

demo.launch()

/tmp/ipykernel_27719/2386466591.py:21: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_27719/2386466591.py:21: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e76a431d75f52923a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
